# 01 — Exploratory Data Analysis

**Goal:** Understand the Amazon Reviews 2023 dataset (Subscription Boxes + Magazine Subscriptions) before any preprocessing or modeling.

**Outputs of this notebook:**
- Dataset overview (size, fields, time range)
- Star rating distribution — overall and per category
- Class imbalance quantification (binary and ternary sentiment framings)
- Review length statistics with appropriate visualizations
- Vocabulary size estimate
- Top discriminative words per sentiment class
- All figures saved to `results/figures/` for the interim presentation

**Relevant for:** Task point 1 of the topic description, RQ1 (class imbalance context), RQ2 (per-category baseline).

In [ ]:
import sys
from pathlib import Path
from collections import Counter
import re
import math

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 150
plt.rcParams['savefig.bbox'] = 'tight'
RANDOM_STATE = 42

FIGURES_DIR = PROJECT_ROOT / 'results' / 'figures'
TABLES_DIR = PROJECT_ROOT / 'results' / 'tables'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

# Color palette: JMU navy with warm contrast
NAVY = '#1E3A6E'
BLUE = '#5B8DBF'
ORANGE = '#D97757'

## 1. Load and combine the two categories

We work with the two smallest categories from the Amazon Reviews 2023 dataset:
- **Subscription Boxes** (~16.2K reviews, 641 items): physical curated boxes (beauty, food, hobbies)
- **Magazine Subscriptions** (~71.5K reviews, 3.4K items): print and digital magazine subscriptions

Both are kept tractable in size, while still being conceptually different enough to enable a meaningful cross-category aspect comparison (RQ2).

In [ ]:
SUBSCRIPTION_BOXES = PROJECT_ROOT / 'data' / 'raw' / 'Subscription_Boxes.jsonl'
MAGAZINE_SUBS = PROJECT_ROOT / 'data' / 'raw' / 'Magazine_Subscriptions.jsonl'

df_boxes = pd.read_json(SUBSCRIPTION_BOXES, lines=True)
df_boxes['category'] = 'Subscription_Boxes'

df_mags = pd.read_json(MAGAZINE_SUBS, lines=True)
df_mags['category'] = 'Magazine_Subscriptions'

df = pd.concat([df_boxes, df_mags], ignore_index=True)
print(f'Total reviews: {len(df):,}')
print(f'\nBy category:')
print(df['category'].value_counts())
df.head(3)

### Dataset structure and time range

In [ ]:
# Convert timestamp (Unix milliseconds) to datetime
df['date'] = pd.to_datetime(df['timestamp'], unit='ms')

print('Columns:', df.columns.tolist())
print(f"\nTime range: {df['date'].min().date()}  ->  {df['date'].max().date()}")
print(f"\nVerified-purchase share: {df['verified_purchase'].mean():.1%}")
print(f"\nMissing values per column:")
print(df[['rating', 'title', 'text']].isna().sum())
print(f"\nEmpty review texts: {(df['text'].str.len() == 0).sum()}")

## 2. Star rating distribution

Star ratings are the natural source for sentiment labels. The distribution directly tells us how (im)balanced our sentiment classes will be — central to RQ1.

In [ ]:
rating_counts = df['rating'].value_counts().sort_index()
rating_pct = (rating_counts / len(df) * 100).round(1)

summary = pd.DataFrame({'count': rating_counts, 'percent': rating_pct})
print('Overall star distribution:')
print(summary)

fig, ax = plt.subplots(figsize=(10, 5.5))
bars = ax.bar(rating_counts.index.astype(int), rating_counts.values,
              color=NAVY, edgecolor='white', linewidth=1.2, width=0.7)
for bar, pct in zip(bars, rating_pct.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1000,
            f'{pct}%', ha='center', va='bottom', fontsize=12, fontweight='bold', color=NAVY)
ax.set_xlabel('Star rating', fontsize=13)
ax.set_ylabel('Number of reviews', fontsize=13)
ax.set_xticks([1, 2, 3, 4, 5])
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.yaxis.set_major_formatter(plt.matplotlib.ticker.FuncFormatter(
    lambda x, _: f'{int(x/1000)}K' if x >= 1000 else str(int(x))))
ax.grid(axis='y', alpha=0.25, linestyle='--')
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'star_distribution_overall.png')
plt.show()

**Interpretation.** The distribution is heavily skewed toward 5-star reviews. Ratings 4–5 dominate, while 2-star ratings are the rarest. This is typical of e-commerce review data — customers who post are disproportionately satisfied, and dissatisfied customers tend to skip middle-ground ratings. Implication for RQ1: any classifier we train will face strong class imbalance, and we cannot rely on accuracy as the headline metric.

### Per-category breakdown

Crucial for RQ2: do both categories show the same imbalance pattern, or do their distributions differ?

In [ ]:
ct_pct = pd.crosstab(df['rating'], df['category'], normalize='columns') * 100
print('Star distribution by category (percent):')
print(ct_pct.round(1))

fig, ax = plt.subplots(figsize=(10, 5.5))
x = np.arange(1, 6)
width = 0.38
ax.bar(x - width/2, ct_pct['Subscription_Boxes'], width,
       label='Subscription Boxes (16K reviews)', color=NAVY, edgecolor='white', linewidth=1)
ax.bar(x + width/2, ct_pct['Magazine_Subscriptions'], width,
       label='Magazine Subscriptions (71K reviews)', color=ORANGE, edgecolor='white', linewidth=1)
ax.set_xlabel('Star rating', fontsize=13)
ax.set_ylabel('Share within category (%)', fontsize=13)
ax.set_xticks(x)
ax.legend(frameon=False, fontsize=11, loc='upper left')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.25, linestyle='--')
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'star_distribution_by_category.png')
plt.show()

**Interpretation.** Both categories share the same overall positive skew, which means the imbalance is a property of customer-review behavior rather than a category-specific artifact. Any classifier built on the combined data will generalize across both categories at least with respect to label distribution. Subtle differences (e.g., one category being slightly more polarized) are worth noting later when interpreting per-category aspect-sentiment profiles in RQ2.

## 3. Class imbalance — quantifying what RQ1 has to deal with

Two common framings of sentiment from star ratings:
- **Binary:** 1–2 stars = negative, 4–5 stars = positive, drop 3-star
- **Ternary:** 1–2 = negative, 3 = neutral, 4–5 = positive

Both views are computed below. The imbalance ratios drive how aggressively we will need class-weighting or resampling in RQ1.

In [ ]:
def map_binary(r):
    if r <= 2:
        return 'negative'
    if r >= 4:
        return 'positive'
    return None  # 3-star excluded

def map_ternary(r):
    if r <= 2:
        return 'negative'
    if r == 3:
        return 'neutral'
    return 'positive'

df['sentiment_binary'] = df['rating'].apply(map_binary)
df['sentiment_ternary'] = df['rating'].apply(map_ternary)

binary_counts = df['sentiment_binary'].value_counts()
ternary_counts = df['sentiment_ternary'].value_counts()

print('Binary framing (3-star reviews dropped):')
print(binary_counts)
ratio = binary_counts['positive'] / binary_counts['negative']
print(f"\n  Imbalance ratio (positive : negative) = {ratio:.2f} : 1")
print(f"  Reviews retained: {binary_counts.sum():,} ({binary_counts.sum() / len(df):.1%} of total)")

print('\n\nTernary framing (all reviews):')
print(ternary_counts)
neg, neu, pos = ternary_counts['negative'], ternary_counts['neutral'], ternary_counts['positive']
print(f'\n  Distribution: {pos/len(df):.1%} positive | {neu/len(df):.1%} neutral | {neg/len(df):.1%} negative')
print(f'  Imbalance: majority class is {pos / neu:.1f}x larger than the smallest (neutral)')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.5))

# Binary
binary_order = ['positive', 'negative']
binary_vals = [binary_counts.get(c, 0) for c in binary_order]
labels1 = ['Positive\n(4-5★)', 'Negative\n(1-2★)']
bars1 = ax1.bar(labels1, binary_vals, color=[NAVY, ORANGE], edgecolor='white', linewidth=1.5, width=0.6)
for bar, v in zip(bars1, binary_vals):
    pct = v / sum(binary_vals) * 100
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(binary_vals)*0.02,
             f'{v:,}\n({pct:.0f}%)', ha='center', fontsize=12, fontweight='bold')
ax1.set_title(f'Binary framing — ratio {binary_vals[0]/binary_vals[1]:.1f} : 1',
              fontsize=13, color=NAVY, fontweight='bold', pad=15)
ax1.set_ylabel('Number of reviews', fontsize=12)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax1.set_ylim(0, max(binary_vals) * 1.18)
ax1.grid(axis='y', alpha=0.25, linestyle='--')
ax1.set_axisbelow(True)

# Ternary
ternary_order = ['positive', 'neutral', 'negative']
ternary_vals = [ternary_counts.get(c, 0) for c in ternary_order]
labels2 = ['Positive\n(4-5★)', 'Neutral\n(3★)', 'Negative\n(1-2★)']
bars2 = ax2.bar(labels2, ternary_vals, color=[NAVY, BLUE, ORANGE], edgecolor='white', linewidth=1.5, width=0.6)
for bar, v in zip(bars2, ternary_vals):
    pct = v / sum(ternary_vals) * 100
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(ternary_vals)*0.02,
             f'{v:,}\n({pct:.0f}%)', ha='center', fontsize=12, fontweight='bold')
ax2.set_title('Ternary framing — neutral is the minority class',
              fontsize=13, color=NAVY, fontweight='bold', pad=15)
ax2.set_ylabel('Number of reviews', fontsize=12)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.set_ylim(0, max(ternary_vals) * 1.18)
ax2.grid(axis='y', alpha=0.25, linestyle='--')
ax2.set_axisbelow(True)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'class_imbalance.png')
plt.show()

**Interpretation.** Two key points for RQ1:

1. **Binary framing yields a moderate imbalance (~3.6 : 1).** This is comfortably handled by class weighting in `LogisticRegression`/`LightGBM`, or by SMOTE for `MultinomialNB` which doesn't support class weights natively. Most published sentiment-analysis benchmarks operate in this regime.
2. **Ternary framing shifts the problem.** Neutral 3-star reviews are only ~8% of the data, making them the minority class — and exactly the population RQ1 wants to investigate (mixed-sentiment reviews). Macro-F1 will be dominated by performance on this class.

**Methodological consequence:** accuracy is misleading here. A classifier that always predicts "positive" already reaches ~78% accuracy under binary framing and ~72% under ternary framing. We will report **macro-F1** and **per-class recall** as the primary metrics, with accuracy only as supplementary.

## 4. Review length analysis

Review length shapes feature engineering decisions: very short reviews carry little signal, very long ones may need truncation for some classifiers.

In [ ]:
df['review_length'] = df['text'].fillna('').str.split().str.len()

print('Overall review length (words):')
print(df['review_length'].describe().round(1))
print(f"\nReviews with 0-5 words: {(df['review_length'] <= 5).sum():,} "
      f"({(df['review_length'] <= 5).mean():.1%})")
print(f"Reviews with 1000+ words: {(df['review_length'] >= 1000).sum():,}")

print('\n\nBy category (review length):')
print(df.groupby('category')['review_length'].agg(['mean', 'median', 'std']).round(1))

In [ ]:
# Histogram with log-scaled y-axis to reveal the long tail
fig, ax = plt.subplots(figsize=(10, 5))
for cat, color in zip(['Subscription_Boxes', 'Magazine_Subscriptions'], [NAVY, ORANGE]):
    subset = df[df['category'] == cat]['review_length'].clip(upper=300)
    ax.hist(subset, bins=60, alpha=0.65, label=cat.replace('_', ' '),
            color=color, edgecolor='white', linewidth=0.4)
ax.set_yscale('log')
ax.set_xlabel('Review length (words, capped at 300 for visibility)', fontsize=12)
ax.set_ylabel('Number of reviews (log scale)', fontsize=12)
ax.legend(frameon=False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'review_length_histogram.png')
plt.show()

**Interpretation.** Review length is heavily right-skewed: the median is around 22 words, the 75th percentile around 47, but the maximum is in the thousands. Most of the signal lives in the first one or two sentences. This has two implications: (1) very short reviews (≤5 words) carry limited information and may be filtered or down-weighted; (2) the short median strongly supports the **sentence-level approach planned for RQ2** — when a review is only 2–3 sentences long, splitting into sentences for aspect attribution is a natural unit, not an artificial one.

## 5. Vocabulary size estimate

A quick whitespace-tokenized vocabulary count gives a sense of feature space dimensionality before any preprocessing. The lowercased version is the realistic upper bound for what TF-IDF will see.

In [ ]:
def quick_tokens(text):
    if not isinstance(text, str):
        return []
    return re.findall(r"[a-zA-Z]+", text.lower())

# Sample to keep it fast; representative since we have 87K reviews
sample = df.sample(n=min(20000, len(df)), random_state=RANDOM_STATE)
vocab_counter = Counter()
total_tokens = 0
for txt in sample['text'].fillna(''):
    toks = quick_tokens(txt)
    vocab_counter.update(toks)
    total_tokens += len(toks)

print(f'Sample size: {len(sample):,} reviews')
print(f'Total tokens: {total_tokens:,}')
print(f'Unique tokens (vocabulary size, raw): {len(vocab_counter):,}')
print(f'Tokens appearing 5+ times: {sum(1 for c in vocab_counter.values() if c >= 5):,}')
print(f'Tokens appearing 20+ times: {sum(1 for c in vocab_counter.values() if c >= 20):,}')
print(f'\n=> After min_df=5 filtering, the TF-IDF vocabulary will be ~{sum(1 for c in vocab_counter.values() if c >= 5):,} unigrams')

**Interpretation.** The TF-IDF vocabulary will land in the low-thousands range after `min_df=5` filtering. This is comfortably manageable for classical models without dimensionality reduction — Logistic Regression and LightGBM both handle features in this regime trivially. With bigrams included, the feature space will grow but stay well under 100K, still tractable.

## 6. Top discriminative words per sentiment class

An early peek at which words distinguish positive from negative reviews. Useful as a sanity check (do they look semantically reasonable?) and as a preview of what TF-IDF + Logistic Regression will likely pick up in RQ1.

In [ ]:
STOPWORDS_QUICK = set('''a an the and or but if of to for in on at by with as is are was were be been being
have has had do does did this that these those i you he she it we they me him her us them my your his their our
not no so just also too very really quite much many some any all only more most less than then now just
would could should can will'''.split())

def top_words(texts, n=20):
    c = Counter()
    for t in texts:
        c.update(w for w in quick_tokens(t) if w not in STOPWORDS_QUICK and len(w) > 2)
    return c.most_common(n)

pos_texts = df[df['sentiment_binary'] == 'positive']['text'].fillna('')
neg_texts = df[df['sentiment_binary'] == 'negative']['text'].fillna('')

top_pos = top_words(pos_texts.sample(n=min(10000, len(pos_texts)), random_state=RANDOM_STATE))
top_neg = top_words(neg_texts.sample(n=min(10000, len(neg_texts)), random_state=RANDOM_STATE))

print('Top 15 words in positive reviews:')
for w, c in top_pos[:15]:
    print(f'  {w:<15} {c:>5,}')
print('\nTop 15 words in negative reviews:')
for w, c in top_neg[:15]:
    print(f'  {w:<15} {c:>5,}')

In [ ]:
# Compute log-odds with Laplace smoothing to find words most associated with each class
pos_dict = dict(top_words(pos_texts.sample(n=min(10000, len(pos_texts)), random_state=RANDOM_STATE), n=300))
neg_dict = dict(top_words(neg_texts.sample(n=min(10000, len(neg_texts)), random_state=RANDOM_STATE), n=300))

all_words = set(pos_dict) | set(neg_dict)
pos_total = sum(pos_dict.values()) + len(all_words)
neg_total = sum(neg_dict.values()) + len(all_words)
log_odds = {}
for w in all_words:
    p_pos = (pos_dict.get(w, 0) + 1) / pos_total
    p_neg = (neg_dict.get(w, 0) + 1) / neg_total
    log_odds[w] = math.log(p_pos / p_neg)

# Filter: word must appear at least 30 times overall (avoid noise)
MIN_FREQ = 30
filtered = {w: lo for w, lo in log_odds.items()
            if (pos_dict.get(w, 0) + neg_dict.get(w, 0)) >= MIN_FREQ}
sorted_lo = sorted(filtered.items(), key=lambda x: x[1])
top_neg_words = sorted_lo[:15]
top_pos_words = sorted_lo[-15:][::-1]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 6))
ax1.barh([w for w, _ in top_pos_words], [lo for _, lo in top_pos_words],
         color=NAVY, edgecolor='white', linewidth=1)
ax1.invert_yaxis()
ax1.set_title('Most positive-leaning words (log-odds)', fontsize=12, color=NAVY, fontweight='bold')
ax1.set_xlabel('Log-odds in favor of positive')
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax1.grid(axis='x', alpha=0.25, linestyle='--')
ax1.set_axisbelow(True)

ax2.barh([w for w, _ in top_neg_words], [-lo for _, lo in top_neg_words],
         color=ORANGE, edgecolor='white', linewidth=1)
ax2.invert_yaxis()
ax2.set_title('Most negative-leaning words (log-odds)', fontsize=12, color=ORANGE, fontweight='bold')
ax2.set_xlabel('Log-odds in favor of negative')
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.grid(axis='x', alpha=0.25, linestyle='--')
ax2.set_axisbelow(True)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'discriminative_words.png')
plt.show()

**Interpretation.** The top discriminative words are semantically reasonable: positive-leaning words like *love*, *great*, *favorite*, *excellent* on one side; negative-leaning words like *cancel*, *refund*, *waste*, *charged*, *terrible* on the other. This is a strong sanity check for two reasons:

1. **The dataset has learnable signal.** A simple log-odds analysis already separates the classes meaningfully, which means a TF-IDF + linear classifier should pick up most of this signal automatically.
2. **Domain-specific patterns emerge.** Words like *cancel*, *refund*, and *charged* are characteristic of subscription products in particular — they relate to billing and renewal experiences. This previews what aspect-level analysis (RQ2) is likely to surface as a major aspect category.

## 7. Summary table for the interim presentation

Compact summary of all key statistics. Saved as CSV for later reference.

In [ ]:
summary_stats = {
    'total_reviews': len(df),
    'subscription_boxes': (df['category'] == 'Subscription_Boxes').sum(),
    'magazine_subscriptions': (df['category'] == 'Magazine_Subscriptions').sum(),
    'time_range': f"{df['date'].min().date()} to {df['date'].max().date()}",
    'verified_purchase_share': f"{df['verified_purchase'].mean():.1%}",
    'mean_review_length_words': round(df['review_length'].mean(), 1),
    'median_review_length_words': int(df['review_length'].median()),
    'star_5_share': f"{(df['rating'] == 5).mean():.1%}",
    'star_1_share': f"{(df['rating'] == 1).mean():.1%}",
    'binary_imbalance_ratio': f"{binary_counts['positive']/binary_counts['negative']:.2f} : 1",
    'ternary_neutral_share': f"{(df['rating'] == 3).mean():.1%}",
    'vocab_unigrams_min_df_5': sum(1 for c in vocab_counter.values() if c >= 5),
}

summary_df = pd.DataFrame([summary_stats]).T.rename(columns={0: 'value'})
summary_df.to_csv(TABLES_DIR / 'eda_summary.csv')
print('Summary saved to results/tables/eda_summary.csv\n')
print(summary_df.to_string())

## 8. Key takeaways for the interim presentation

**Dataset scale and composition**
- Combined dataset of **~87.7K reviews** across two distinct categories. Subscription Boxes (~16K) is the smallest standalone category in the entire Amazon Reviews 2023 release; pairing it with Magazine Subscriptions (~71K) keeps the analysis tractable while enabling the cross-category comparison required by the topic description.
- Both categories cover a long time range (1996–2023), so any modeling will need a chronological train/test split to avoid temporal leakage.

**Strong class imbalance — RQ1 framing**
- The star distribution is heavily skewed toward 5-star reviews (≈60%), with only ~14% 1-star and ~8% 3-star.
- Under a **binary framing** (drop 3-star), the positive-to-negative ratio is roughly **3.6 : 1** — moderately imbalanced, manageable with class weighting or SMOTE.
- Under a **ternary framing**, the **neutral 3-star class is the minority** at only ~8% of all reviews. This will be the hardest class to classify and is exactly the population RQ1 wants to investigate (mixed-sentiment reviews).

**Implications for evaluation (RQ1)**
- Reporting accuracy alone would mask poor performance on the minority class. We will report **macro-F1** and **per-class recall** as primary metrics.
- The skew also implies that a trivial "always predict positive" baseline already reaches ~78% accuracy — any classifier must be benchmarked against that reference.

**Review length and vocabulary**
- Reviews are short on average (median 22 words) but heavily right-skewed (max ≈4900 words). Most signal lives in the first couple of sentences, which supports the sentence-level approach planned for RQ2.
- The TF-IDF vocabulary will be in the **low-thousands** range after `min_df=5` filtering — comfortable for classical models without dimensionality reduction.

**Discriminative-word sanity check**
- Top positive-leaning words (e.g. *love*, *great*, *favorite*) and negative-leaning words (e.g. *cancel*, *waste*, *refund*, *charged*) are semantically reasonable and aligned with what a TF-IDF + linear model is expected to pick up. This indicates the dataset has learnable signal — there is no obvious data-quality showstopper.

**Open points to raise in the interim presentation**
- **Sentiment framing decision:** binary (drop 3-star) vs. ternary. Binary is methodologically cleaner; ternary engages directly with RQ1's stated focus on "mixed-sentiment 3-star reviews." *Will be raised with the supervisor.*
- **Aspect extraction approach:** keyword lexicon vs. noun-phrase chunking. *Will be raised with the supervisor.*
- **Verified-purchase filter:** should we restrict to verified-purchase reviews only, to reduce noise from spam or incentivized reviews?
- **Item-level concentration in Subscription Boxes:** with only 641 items, aspect frequencies may be dominated by a few popular boxes. Worth flagging.